# 01 - Data Exploration: Synthetic Enterprise Corpus

This notebook explores the synthetic enterprise corpus generated for the Indirect Prompt Injection study.

The corpus simulates an enterprise knowledge base composed of:
- **Confluence pages** -- internal wiki-style documentation
- **Slack messages** -- team chat messages
- **Emails** -- internal/external email communications
- **Internal docs** -- formal company documents (policies, procedures, guides)

Each document carries rich metadata: `source_type`, `author`, `department`, `access_level`, `trust_score`, and `created_at`.

A subset of documents (~20) have been deliberately **poisoned** with indirect prompt injection payloads across 4 attack categories:
1. Data Exfiltration
2. Phishing
3. Goal Hijacking
4. Privilege Escalation

The ground-truth manifest (`manifest.json`) tracks exactly which documents are poisoned, enabling accurate evaluation.

In [ ]:
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

from config.settings import settings

print(f"Project root: {PROJECT_ROOT}")
print(f"Data output dir: {settings.data_output_dir}")

## 1. Load the Corpus and Manifest

The data generation script (`scripts/generate_data.py`) produces two files:
- `corpus.json` -- all documents (clean + poisoned)
- `manifest.json` -- ground truth of which documents contain injected payloads

In [ ]:
corpus_path = Path(settings.data_output_dir) / "corpus.json"
manifest_path = Path(settings.data_output_dir) / "manifest.json"

with open(corpus_path, "r") as f:
    corpus = json.load(f)

with open(manifest_path, "r") as f:
    manifest = json.load(f)

# Convert to DataFrames for easy analysis
df = pd.DataFrame(corpus)
manifest_df = pd.DataFrame(manifest)

print(f"Total documents in corpus: {len(corpus)}")
print(f"Poisoned documents (manifest): {len(manifest)}")
print(f"Clean documents: {len(corpus) - len(manifest)}")
print(f"\nCorpus columns: {list(df.columns)}")

## 2. Basic Statistics

In [ ]:
print("=" * 60)
print("CORPUS STATISTICS")
print("=" * 60)

# Documents by source type
print("\n--- Documents by Source Type ---")
source_counts = df["source_type"].value_counts()
for source, count in source_counts.items():
    print(f"  {source:20s}: {count:4d} ({count/len(df)*100:.1f}%)")

# Documents by access level
print("\n--- Documents by Access Level ---")
access_counts = df["access_level"].value_counts()
for level, count in access_counts.items():
    print(f"  {level:20s}: {count:4d} ({count/len(df)*100:.1f}%)")

# Documents by department
print("\n--- Documents by Department ---")
dept_counts = df["department"].value_counts()
for dept, count in dept_counts.items():
    print(f"  {dept:20s}: {count:4d}")

# Trust score statistics
print("\n--- Trust Score Statistics ---")
print(f"  Mean:   {df['trust_score'].mean():.3f}")
print(f"  Median: {df['trust_score'].median():.3f}")
print(f"  Std:    {df['trust_score'].std():.3f}")
print(f"  Min:    {df['trust_score'].min():.3f}")
print(f"  Max:    {df['trust_score'].max():.3f}")

# Content length statistics
df["content_length"] = df["content"].str.len()
print("\n--- Content Length (chars) ---")
print(f"  Mean:   {df['content_length'].mean():.0f}")
print(f"  Median: {df['content_length'].median():.0f}")
print(f"  Min:    {df['content_length'].min():.0f}")
print(f"  Max:    {df['content_length'].max():.0f}")

## 3. Example Documents by Source Type

Let us inspect a sample document from each source type to understand the content format.

In [ ]:
for source_type in df["source_type"].unique():
    sample = df[df["source_type"] == source_type].iloc[0]
    print("=" * 70)
    print(f"SOURCE TYPE: {source_type.upper()}")
    print("=" * 70)
    print(f"  ID:           {sample.get('id', 'N/A')}")
    print(f"  Title:        {sample.get('title', 'N/A')}")
    print(f"  Author:       {sample.get('author', 'N/A')}")
    print(f"  Department:   {sample.get('department', 'N/A')}")
    print(f"  Access Level: {sample.get('access_level', 'N/A')}")
    print(f"  Trust Score:  {sample.get('trust_score', 'N/A')}")
    print(f"  Created At:   {sample.get('created_at', 'N/A')}")
    print(f"\n  Content (first 500 chars):")
    content = sample.get("content", "")
    print(f"  {content[:500]}")
    if len(content) > 500:
        print(f"  ... [{len(content) - 500} more chars]")
    print()

## 4. Metadata Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Corpus Metadata Distributions", fontsize=16, fontweight="bold")

# --- Source Type Pie Chart ---
ax = axes[0, 0]
source_counts = df["source_type"].value_counts()
colors = sns.color_palette("Set2", len(source_counts))
wedges, texts, autotexts = ax.pie(
    source_counts.values,
    labels=source_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
)
ax.set_title("Documents by Source Type")

# --- Access Level Bar Chart ---
ax = axes[0, 1]
access_order = ["public", "internal", "confidential", "restricted"]
access_counts = df["access_level"].value_counts()
# Reindex to desired order, filling missing levels with 0
access_counts = access_counts.reindex(access_order, fill_value=0)
bar_colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]
ax.bar(access_counts.index, access_counts.values, color=bar_colors)
ax.set_title("Documents by Access Level")
ax.set_ylabel("Count")
for i, v in enumerate(access_counts.values):
    ax.text(i, v + 2, str(v), ha="center", fontweight="bold")

# --- Trust Score Histogram ---
ax = axes[1, 0]
ax.hist(df["trust_score"], bins=20, edgecolor="black", alpha=0.7, color="#5C6BC0")
ax.axvline(df["trust_score"].mean(), color="red", linestyle="--", label=f"Mean = {df['trust_score'].mean():.2f}")
ax.set_title("Trust Score Distribution")
ax.set_xlabel("Trust Score")
ax.set_ylabel("Count")
ax.legend()

# --- Department Distribution ---
ax = axes[1, 1]
dept_counts = df["department"].value_counts()
ax.barh(dept_counts.index, dept_counts.values, color=sns.color_palette("viridis", len(dept_counts)))
ax.set_title("Documents by Department")
ax.set_xlabel("Count")

plt.tight_layout()
plt.show()

## 5. Poisoned Documents Analysis

The manifest records every poisoned document along with its attack type, variant, and success patterns.
Let us inspect the injected content.

In [ ]:
# Overview of poisoned documents
print(f"Total poisoned documents: {len(manifest)}")
print("\n--- Poisoned Docs by Attack Type ---")
attack_type_counts = manifest_df["attack_type"].value_counts()
for atype, count in attack_type_counts.items():
    print(f"  {atype:25s}: {count}")

print("\n--- Poisoned Docs by Source Type ---")
if "source_type" in manifest_df.columns:
    for stype, count in manifest_df["source_type"].value_counts().items():
        print(f"  {stype:25s}: {count}")

In [ ]:
# Build a set of poisoned doc IDs for easy lookup
poisoned_ids = set(manifest_df["doc_id"].tolist()) if "doc_id" in manifest_df.columns else set(manifest_df["id"].tolist())

# Show example poisoned documents with the injected content highlighted
for i, entry in enumerate(manifest[:4]):  # one per attack type
    doc_id = entry.get("doc_id", entry.get("id", "unknown"))
    attack_type = entry.get("attack_type", "unknown")
    
    # Find the full document in the corpus
    matching_docs = [d for d in corpus if d.get("id") == doc_id]
    
    print("=" * 70)
    print(f"POISONED DOC #{i+1} -- Attack: {attack_type.upper()}")
    print("=" * 70)
    print(f"  Doc ID:       {doc_id}")
    print(f"  Attack Type:  {attack_type}")
    print(f"  Variant:      {entry.get('variant', entry.get('variant_id', 'N/A'))}")
    
    if "success_patterns" in entry:
        print(f"  Success Patterns: {entry['success_patterns'][:3]}")
    
    if matching_docs:
        doc = matching_docs[0]
        print(f"  Source Type:  {doc.get('source_type', 'N/A')}")
        print(f"  Title:        {doc.get('title', 'N/A')}")
        print(f"  Access Level: {doc.get('access_level', 'N/A')}")
        print(f"  Trust Score:  {doc.get('trust_score', 'N/A')}")
        content = doc.get("content", "")
        print(f"\n  Content (first 600 chars):")
        print(f"  >>> {content[:600]}")
        if len(content) > 600:
            print(f"  ... [{len(content) - 600} more chars]")
    else:
        print("  [Document not found in corpus by ID]")
    print()

## 6. Trust Score Distribution: Clean vs Poisoned

Do poisoned documents have different trust score characteristics compared to clean documents?
This comparison helps motivate the Source Trust Scoring defense.

In [ ]:
# Tag each document as clean or poisoned
id_col = "doc_id" if "doc_id" in manifest_df.columns else "id"
poisoned_ids = set(manifest_df[id_col].tolist())
df["is_poisoned"] = df["id"].isin(poisoned_ids)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Overlaid Histograms ---
ax = axes[0]
clean_scores = df[~df["is_poisoned"]]["trust_score"]
poisoned_scores = df[df["is_poisoned"]]["trust_score"]

ax.hist(clean_scores, bins=20, alpha=0.6, label=f"Clean (n={len(clean_scores)})", color="#4CAF50", edgecolor="black")
ax.hist(poisoned_scores, bins=10, alpha=0.8, label=f"Poisoned (n={len(poisoned_scores)})", color="#F44336", edgecolor="black")
ax.set_title("Trust Score: Clean vs Poisoned")
ax.set_xlabel("Trust Score")
ax.set_ylabel("Count")
ax.legend()

# --- Box Plot ---
ax = axes[1]
data_for_box = [
    clean_scores.values,
    poisoned_scores.values,
]
bp = ax.boxplot(data_for_box, labels=["Clean", "Poisoned"], patch_artist=True)
bp["boxes"][0].set_facecolor("#4CAF50")
bp["boxes"][1].set_facecolor("#F44336")
ax.set_title("Trust Score Box Plot")
ax.set_ylabel("Trust Score")

plt.suptitle("Trust Score Comparison: Clean vs Poisoned Documents", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
print("\n--- Trust Score Summary ---")
print(f"  Clean docs:    mean={clean_scores.mean():.3f}, median={clean_scores.median():.3f}")
print(f"  Poisoned docs: mean={poisoned_scores.mean():.3f}, median={poisoned_scores.median():.3f}")

In [ ]:
# Source type breakdown for poisoned vs clean
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
clean_sources = df[~df["is_poisoned"]]["source_type"].value_counts()
ax.pie(clean_sources.values, labels=clean_sources.index, autopct="%1.1f%%",
       colors=sns.color_palette("Set2", len(clean_sources)), startangle=90)
ax.set_title("Clean Documents by Source Type")

ax = axes[1]
poisoned_sources = df[df["is_poisoned"]]["source_type"].value_counts()
ax.pie(poisoned_sources.values, labels=poisoned_sources.index, autopct="%1.1f%%",
       colors=sns.color_palette("Set1", len(poisoned_sources)), startangle=90)
ax.set_title("Poisoned Documents by Source Type")

plt.suptitle("Source Type Distribution: Clean vs Poisoned", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

Key observations from the data exploration:

1. **Corpus composition**: The corpus contains ~500+ documents distributed across 4 source types (Confluence, Slack, Email, Internal Docs) with realistic metadata.

2. **Poisoned documents**: ~20 documents have been injected with attack payloads (5 per attack type). These are intentionally spread across different source types to simulate realistic attack vectors.

3. **Trust score distribution**: Poisoned documents may have varying trust scores depending on the attack strategy. Some attacks deliberately target high-trust source types to evade trust-based defenses.

4. **Access levels**: The corpus includes documents at all access levels (public, internal, confidential, restricted), which is critical for testing the Privilege Escalation attack and the Privilege Filter defense.

Next: See **02_attack_demo.ipynb** to observe these poisoned documents enabling successful attacks against the undefended RAG pipeline.